# Crypto.Net · 08 — Inti Rust vs C# managed

*Dibuat oleh Gravicode Studios dipimpin oleh Kang Fadhil.*

> **Aman dijalankan.** Notebook ini hanya membaca jaringan publik dan menandatangani secara offline; tidak pernah mem-broadcast transaksi. Kunci yang dibuat di sini adalah kunci demo — jangan pernah mengirim dana sungguhan ke sana.

Buka di VS Code dengan ekstensi **Polyglot Notebooks** (atau Jupyter dengan .NET Interactive) lalu jalankan sel secara berurutan. [English version](./08-benchmarks.en.ipynb)

Yang akan Anda lakukan: menjalankan benchmark bawaan yang mengukur setiap primitif di kedua mesin. Anda juga akan mengukur beban kerja sendiri dengan `CryptoNative.UseBackend`, memastikan kedua mesin menghasilkan byte yang identik, dan melihat mengapa `Auto` mempertahankan SHA-2 di implementasi platform yang dipercepat hardware.

In [ ]:
#r "nuget: Crypto.Net.Extensions, 1.2.0"

In [ ]:
using System.Diagnostics;
using Crypto.Net.Native;
using Crypto.Net.Extensions;

$"Rust core: {NativeLoader.IsNativeAvailable} ({NativeLoader.RuntimeIdentifier}), default backend {CryptoNative.Backend}"

## Benchmark bawaan

`CryptoBenchmark.Run` mengukur operasi per detik di setiap mesin. `cnet bench` dan grafik Gallery memakai benchmark yang sama. `scale` memperpendek atau memperpanjang setiap pengukuran.

In [ ]:
var results = CryptoBenchmark.Run(scale: 0.3);
results.Select(r => new
{
    r.Category,
    r.Name,
    Rust = $"{r.NativeOpsPerSec:N0}/s",
    Managed = $"{r.ManagedOpsPerSec:N0}/s",
    Speedup = r.Speedup > 0 ? $"{r.Speedup:0.0}×" : "n/a",
})

## Beban kerja Anda sendiri

`CryptoNative.UseBackend` mengganti mesin hanya untuk alur async saat ini, sehingga Anda bisa membandingkan di dalam satu proses, atau mengunci sebuah request ke mesin managed.

In [ ]:
byte[] key = Convert.FromHexString("4c0883a69102937d6231471b5dbb6204fe5129617082792ae468d01a3f362318");
byte[] digest = CryptoNative.Keccak256("hello"u8);

double Time(CryptoBackend backend, int n)
{
    using var _ = CryptoNative.UseBackend(backend);
    var sw = Stopwatch.StartNew();
    for (int i = 0; i < n; i++) CryptoNative.Secp256k1SignRecoverable(key, digest);
    return n / sw.Elapsed.TotalSeconds;
}
Time(CryptoBackend.Native, 50); Time(CryptoBackend.Managed, 50);   // warm up
double native = Time(CryptoBackend.Native, 2000), managed = Time(CryptoBackend.Managed, 2000);
$"ECDSA sign: Rust {native:N0}/s · managed {managed:N0}/s · {native / managed:0.0}× faster"

## Hasil yang sama

Mesin managed adalah fallback, bukan pendekatan: CI menjalankan test kesetaraan untuk setiap primitif. Anda bisa memeriksanya sendiri di sini.

In [ ]:
(byte[] Sig, byte Rec) Sign(CryptoBackend b) { using var _ = CryptoNative.UseBackend(b); return CryptoNative.Secp256k1SignRecoverable(key, digest); }
byte[] Blake(CryptoBackend b) { using var _ = CryptoNative.UseBackend(b); return CryptoNative.Blake2b256("hello"u8); }
new
{
    EcdsaIdentical = Sign(CryptoBackend.Native).Sig.SequenceEqual(Sign(CryptoBackend.Managed).Sig),
    Blake2bIdentical = Blake(CryptoBackend.Native).SequenceEqual(Blake(CryptoBackend.Managed)),
}

## Mengapa `Auto` mempertahankan SHA-2 di platform

SHA-256 dan HMAC-SHA512 di .NET memakai ekstensi SHA CPU lewat OS. Untuk input kecil, biaya memanggil Rust lewat batas FFI lebih besar daripada yang dihemat Rust. Karena itu `Auto` memakai Rust untuk semuanya kecuali SHA-2, HMAC dan PBKDF2.

In [ ]:
double Sha(CryptoBackend b)
{
    using var _ = CryptoNative.UseBackend(b);
    var sw = Stopwatch.StartNew();
    for (int i = 0; i < 200_000; i++) CryptoNative.Sha256(digest);
    return 200_000 / sw.Elapsed.TotalSeconds;
}
Sha(CryptoBackend.Auto);
$"SHA-256 (32 bytes): Auto/platform {Sha(CryptoBackend.Auto):N0}/s · Rust {Sha(CryptoBackend.Native):N0}/s"